In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run util path

In [0]:
dbutils.widgets.text("catalog", "ecommerce", "Catalog")
dbutils.widgets.text("data_source", "sellers", "Data Source")

In [0]:
catalog= dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
orders=spark.sql(f"select * from {catalog}.{gold_schema}.sb_dim_orders")
customers=spark.sql(f"select * from {catalog}.{gold_schema}. sb_dim_customers")
payments=spark.sql(f"select * from {catalog}.{gold_schema}.sb_dim_payments")
order_items=spark.sql(f"select * from {catalog}.{gold_schema}.sb_dim_order_items")
sellers=spark.sql(f"select * from {catalog}.{gold_schema}.sb_dim_sellers")
products=spark.sql(f"select * from {catalog}.{gold_schema}.sb_dim_products")

Aggeregate one-to-many-tables

In [0]:
order_items_full = order_items \
.join(products, "product_id", "left") \
.join(sellers, "seller_id", "left")

In [0]:
order_items_full.columns

In [0]:
order_items_features = order_items_full.groupBy("order_id").agg(

    F.count("*").alias("item_count"),

    F.round(F.sum("price"),2).alias("total_price"),

    F.round(F.sum("freight_value"),2).alias("total_freight"),

    F.first("seller_city").alias("seller_city"),

    F.first("seller_state").alias("seller_state")
)

In [0]:
payment_features=payments

Join the tables

In [0]:
df = orders \
.join(customers,"customer_id","left") \
.join(order_items_features,"order_id","left") \
.join(payment_features,"order_id","left") 

Final feature engineering


In [0]:
df.columns

In [0]:
df = df.withColumn(
    "same_state_delivery",
    F.when(F.col("customer_state")==F.col("seller_state"),1).otherwise(0)
)

In [0]:
df = df.withColumn(
    "same_city_delivery",
    F.when(F.col("customer_city")==F.col("seller_city"),1).otherwise(0)
)

In [0]:
df = df.withColumn(
"freight_ratio",
F.when(F.col("total_price") > 0,
       F.col("total_freight") / F.col("total_price")
).otherwise(0)
)

Data Validation

In [0]:
df.groupBy("order_id") \
.count() \
.filter("count > 1") \
.display()

In [0]:
df_1 = df.filter(F.col("order_status") == "delivered")

In [0]:
df_1.select([
F.sum(F.col(c).isNull().cast("int")).alias(c)
for c in df.columns
]).display()

In [0]:
df_1 = df_1.fillna({
    "payment_amount": 0,
    "installments": 0
})
df_1.select([
F.sum(F.col(c).isNull().cast("int")).alias(c)
for c in df.columns
]).display()

In [0]:
df_1.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.feature_engineered_data")